
# Notebook de treinamento da(s) baseline(s)

Problema de negócios: qual a probabilidade de um cliente reativar nos próximos 30 dias?

In [0]:
%pip install xgboost optuna
%restart_python

In [0]:
#---- Libs

from pyspark.sql.functions import *
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from mlflow.models import infer_signature
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
import math
from builtins import max as max_builtins


#---- Tabelas

SOURCE_TABLE = 'workspace.gold_layer.customer_reactivation_modeling'

#---- Parâmetros

## Datas de treino

START_DATE_TRAIN = '2010-03-08'
END_DATE_TRAIN = '2011-05-08'

## Datas de validação

START_DATE_VALIDATION = '2011-06-08'
END_DATE_VALIDATION = '2011-08-08'

### Datas de teste

START_DATE_TEST = '2011-09-08'
END_DATE_TEST = '2011-11-08'

## Random state
RANDOM_STATE = 19

In [0]:
#---- Criação do experimento

EXPERIMENT_NAME = '/Users/lul.rafaelbarbosa@gmail.com/reactivation-customers-experiment'

mlflow.set_experiment(EXPERIMENT_NAME)

In [0]:
df = spark.sql(
    f'''
    select * from {SOURCE_TABLE}
    '''
)

df.display()

In [0]:
df.groupBy('target_reactivated_30d').agg(
    count('*').alias('count'),
    ((count('*') / df.count() * 100)).alias('percent')
).display()


## Separando em treino e teste

In [0]:
df_train = (
    df
    .filter((col('reference_date') >= START_DATE_TRAIN) & (col('reference_date') <= END_DATE_TRAIN))
)

df_valid = (
    df
    .filter((col('reference_date') >= START_DATE_VALIDATION) & (col('reference_date') <= END_DATE_VALIDATION))
)

df_test = (
    df
    .filter((col('reference_date') >= START_DATE_TEST) & (col('reference_date') <= END_DATE_TEST))
)

In [0]:
#---- Validações: 

print('Dataset de treino')
df_train.agg(min('reference_date').alias('min'), max('reference_date').alias('max'), count('*').alias('count')).display()

print('Dataset de validação')
df_valid.agg(min('reference_date').alias('min'), max('reference_date').alias('max'), count('*').alias('count')).display()

print('Dataset de test')
df_test.agg(min('reference_date').alias('min'), max('reference_date').alias('max'), count('*').alias('count')).display()

In [0]:
#---- Convertendo para o Pandas

df_train_pandas = df_train.toPandas()
df_valid_pandas = df_valid.toPandas()
df_test_pandas = df_test.toPandas()

In [0]:
from decimal import Decimal


def convert_decimal_to_float(dataframe):
    dataframe = dataframe.copy()

    for column_name in dataframe.columns:
        contains_decimal = (
            dataframe[column_name]
            .dropna()
            .map(lambda value: isinstance(value, Decimal))
            .any()
        )

        if contains_decimal:
            dataframe[column_name] = dataframe[column_name].astype("float64")

    return dataframe


df_train_pandas = convert_decimal_to_float(df_train_pandas)
df_valid_pandas = convert_decimal_to_float(df_valid_pandas)
df_test_pandas = convert_decimal_to_float(df_test_pandas)

In [0]:
TARGET_COLUMN = "target_reactivated_30d"

FEATURE_COLUMNS = [
    "inactive_days",
    "number_orders",
    "total_items",
    "total_spent",
    "unique_products",
    "distinct_countries",
    "customer_tenure_days",
    "average_ticket",
    "average_amount_per_country",
    "average_amount_per_stock_code",
]

IDENTIFIER_COLUMNS = [
    "customer_id",
    "reference_date"
]

X_train = df_train_pandas[FEATURE_COLUMNS]
y_train = df_train_pandas[TARGET_COLUMN]

X_valid = df_valid_pandas[FEATURE_COLUMNS]
y_valid = df_valid_pandas[TARGET_COLUMN]

X_test = df_test_pandas[FEATURE_COLUMNS]
y_test = df_test_pandas[TARGET_COLUMN]


## Métricas

In [0]:
#---- Parâmetros: 

TOP_K_VALUES = [0.10, 0.20, 0.30]

In [0]:
def calculate_top_k_metrics(
    y_true, y_score, dataset_name, top_k_values=TOP_K_VALUES, random_state=RANDOM_STATE
):
    ranking_df = pd.DataFrame(
        {"target": (np.asarray(y_true)), "score": (np.asarray(y_score))}
    )

    random_generator = np.random.default_rng(random_state)

    ranking_df["tie_breaker"] = random_generator.random(len(ranking_df))

    ranking_df = ranking_df.sort_values(
        by=["score", "tie_breaker"], ascending=[False, True], kind="mergesort"
    ).reset_index(drop=True)

    positive_rate = ranking_df["target"].mean()

    total_positives = ranking_df["target"].sum()

    metrics = {}

    for top_k in top_k_values:
        percentage = int(top_k * 100)

        selected_rows = max_builtins(1, math.ceil(len(ranking_df) * top_k))

        top_population = ranking_df.head(selected_rows)

        precision_at_k = top_population["target"].mean()

        recall_at_k = top_population["target"].sum() / total_positives

        lift_at_k = precision_at_k / positive_rate

        metrics[f"{dataset_name}_precision_at_{percentage}pct"] = float(precision_at_k)

        metrics[f"{dataset_name}_recall_at_{percentage}pct"] = float(recall_at_k)

        metrics[f"{dataset_name}_lift_at_{percentage}pct"] = float(lift_at_k)

    return metrics

In [0]:
def evaluate_classifier(model, X, y, dataset_name):
    # Classes previstas
    y_pred = model.predict(X)

    # Localiza a classe positiva
    positive_class_index = np.where(model.classes_ == 1)[0][0]

    # Probabilidade da classe positiva
    y_proba = model.predict_proba(X)[:, positive_class_index]

    # Matriz de confusão
    tn, fp, fn, tp = confusion_matrix(y_true=y, y_pred=y_pred, labels=[0, 1]).ravel()

    metrics = {
        # Distribuição real
        f"{dataset_name}_positive_rate": float(np.mean(y)),
        # Distribuição predita
        f"{dataset_name}_predicted_positive_rate": float(np.mean(y_pred)),
        # Principal métrica técnica
        f"{dataset_name}_pr_auc": float(
            average_precision_score(y_true=y, y_score=y_proba)
        ),
        # Métrica geral de ranking
        f"{dataset_name}_roc_auc": float(roc_auc_score(y_true=y, y_score=y_proba)),
        # Qualidade das probabilidades
        f"{dataset_name}_brier_score": float(brier_score_loss(y, y_proba, pos_label=1)),
        # Métricas no resultado do predict()
        f"{dataset_name}_precision": float(
            precision_score(y_true=y, y_pred=y_pred, zero_division=0)
        ),
        f"{dataset_name}_recall": float(
            recall_score(y_true=y, y_pred=y_pred, zero_division=0)
        ),
        f"{dataset_name}_f1": float(f1_score(y_true=y, y_pred=y_pred, zero_division=0)),
        # Matriz de confusão
        f"{dataset_name}_true_negative": float(tn),
        f"{dataset_name}_false_positive": float(fp),
        f"{dataset_name}_false_negative": float(fn),
        f"{dataset_name}_true_positive": float(tp),
    }

    top_k_metrics = calculate_top_k_metrics(
        y_true=y, y_score=y_proba, dataset_name=(dataset_name)
    )

    metrics.update(top_k_metrics)

    confusion_matrix_data = {
        "true_negative": (int(tn)),
        "false_positive": (int(fp)),
        "false_negative": (int(fn)),
        "true_positive": (int(tp)),
    }

    return (metrics, confusion_matrix_data)

In [0]:
def display_confusion_matrix(confusion_data, dataset_name):
    confusion_df = pd.DataFrame(
        [
            [confusion_data["true_negative"], confusion_data["false_positive"]],
            [confusion_data["false_negative"], confusion_data["true_positive"]],
        ],
        index=["Real = 0", "Real = 1"],
        columns=["Predito = 0", "Predito = 1"],
    )

    print(
        f"""
        Matriz de confusão:

        {dataset_name}
        """
    )

    display(confusion_df)


## XGBoost Classifier + Class Weights

In [0]:
from xgboost import XGBClassifier 

#---- PARÂMETROS:

RUN_NAME = "simple_xgboost_classifier"

In [0]:
#---- Peso da classe 0 

scale_pos_weight = (
    df_train_pandas.target_reactivated_30d.value_counts(normalize=True)[0] / df_train_pandas.target_reactivated_30d.value_counts(normalize=True)[1]
)

scale_pos_weight

In [0]:
#---- Primeira versão: relativamente aleatória

xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=19,
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    n_jobs=-1
)

In [0]:
with mlflow.start_run(run_name=RUN_NAME) as run:
    # ==========================
    # Treinamento
    # ==========================

    xgb_model.fit(X_train, y_train)

    # ==========================
    # Avaliação
    # ==========================

    train_metrics, train_confusion = evaluate_classifier(
        model=xgb_model, X=X_train, y=y_train, dataset_name="train"
    )

    valid_metrics, valid_confusion = evaluate_classifier(
        model=xgb_model, X=X_valid, y=y_valid, dataset_name="valid"
    )

    test_metrics, test_confusion = evaluate_classifier(
        model=xgb_model, X=X_test, y=y_test, dataset_name="test"
    )

    all_metrics = {**train_metrics, **valid_metrics, **test_metrics}

    # ==========================
    # Parâmetros
    # ==========================

    mlflow.log_params(
        {
            "model_class": ("XGBoostClassifier"),
            "strategy": (None),
            "target_column": (TARGET_COLUMN),
            "number_features": (len(FEATURE_COLUMNS)),
            "feature_columns": (", ".join(FEATURE_COLUMNS)),
            "top_k_values": ("10%, 20%, 30%"),
            "top_k_tie_breaker": ("random"),
            "top_k_random_state": (RANDOM_STATE),
            "train_start_date": (str(df_train_pandas["reference_date"].min())),
            "train_end_date": (str(df_train_pandas["reference_date"].max())),
            "valid_start_date": (str(df_valid_pandas["reference_date"].min())),
            "valid_end_date": (str(df_valid_pandas["reference_date"].max())),
            "test_start_date": (str(df_test_pandas["reference_date"].min())),
            "test_end_date": (str(df_test_pandas["reference_date"].max())),
        }
    )

    # ==========================
    # Volumes
    # ==========================

    mlflow.log_metrics(
        {
            "train_rows": float(len(X_train)),
            "valid_rows": float(len(X_valid)),
            "test_rows": float(len(X_test)),
        }
    )

    # ==========================
    # Métricas
    # ==========================

    mlflow.log_metrics(all_metrics)

    # ==========================
    # Tags
    # ==========================

    mlflow.set_tags(
        {
            "project": "reactivation-customers-model",
            "model_type": "candidate",
            "model_family": "xgboost",
            "framework": "xgboost",
            "optimization_library": "optuna",
            "class_imbalance_strategy": "scale_pos_weight",
            "dataset": ("workspace.gold_layer.customer_reactivation_modeling"),
        }
    )

    # ==========================
    # Matriz de confusão
    # ==========================

    mlflow.log_dict(
        {
            "train": (train_confusion),
            "valid": (valid_confusion),
            "test": (test_confusion),
        },
        "evaluation/confusion_matrices.json",
    )

    # ==========================
    # Metadados das features
    # ==========================

    mlflow.log_dict(
        {
            "identifier_columns": (IDENTIFIER_COLUMNS),
            "feature_columns": (FEATURE_COLUMNS),
            "target_column": (TARGET_COLUMN),
        },
        "metadata/model_columns.json",
    )

    # ==========================
    # Assinatura do modelo
    # ==========================

    model_signature = infer_signature(
        model_input=(X_train), model_output=(xgb_model.predict(X_train))
    )

    # ==========================
    # Artefato do modelo
    # ==========================

    mlflow.sklearn.log_model(
        sk_model=(xgb_model),
        name=("model"),
        signature=(model_signature),
        input_example=(X_train.head(5)),
    )

    print(
        f"""
        Run registrada com sucesso.

        Run ID:

        {run.info.run_id}
        """
    )

In [0]:
metrics_comparison = pd.DataFrame(
    [
        {
            "dataset": ("train"),
            **{
                metric.replace("train_", ""): value
                for metric, value in train_metrics.items()
            },
        },
        {
            "dataset": ("valid"),
            **{
                metric.replace("valid_", ""): value
                for metric, value in valid_metrics.items()
            },
        },
        {
            "dataset": ("test"),
            **{
                metric.replace("test_", ""): value
                for metric, value in test_metrics.items()
            },
        },
    ]
)

display(metrics_comparison)

In [0]:
y_train_pred = xgb_model.predict(X_train)
y_valid_pred = xgb_model.predict(X_valid)
y_test_pred = xgb_model.predict(X_test)

In [0]:
from IPython.display import display as pandas_display

def display_confusion_matrix(y_true, y_pred, dataset_name):
    confusion_df = (
        pd.crosstab(
            pd.Series(y_true, name="Valor real"),
            pd.Series(y_pred, name="Valor predito"),
            dropna=False,
        )
        # Garante que as classes 0 e 1
        # apareçam mesmo quando o modelo
        # não prevê uma delas
        .reindex(index=[0, 1], columns=[0, 1], fill_value=0)
        .rename(
            index={0: "Real = 0", 1: "Real = 1"},
            columns={0: "Predito = 0", 1: "Predito = 1"},
        )
    )

    print(f"Matriz de confusão — {dataset_name}")

    pandas_display(confusion_df)

In [0]:
display_confusion_matrix(y_true=y_train, y_pred=y_train_pred, dataset_name="Train")

display_confusion_matrix(y_true=y_valid, y_pred=y_valid_pred, dataset_name="Validation")

display_confusion_matrix(y_true=y_test, y_pred=y_test_pred, dataset_name="Test")


## XGBoost + Class Weights + Optuna 

In [0]:
# ==========================================
# PESO DA CLASSE POSITIVA
# ==========================================

negative_class_count = int((y_train == 0).sum())

positive_class_count = int((y_train == 1).sum())


if positive_class_count == 0:
    raise ValueError(
        """
        Não existem observações da classe positiva
        no conjunto de treino.
        """
    )


scale_pos_weight = negative_class_count / positive_class_count


print(
    f"""
    Quantidade da classe 0:
    {negative_class_count:,}

    Quantidade da classe 1:
    {positive_class_count:,}

    Scale Pos Weight:
    {scale_pos_weight:.4f}
    """
)

In [0]:
import optuna

# ==========================================
# CONFIGURAÇÕES
# ==========================================

RUN_NAME = "optuna_xgboost_classifier"
N_TRIALS = 100

# ==========================================
# PARÂMETROS FIXOS
# ==========================================

FIXED_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "random_state": RANDOM_STATE,
    "scale_pos_weight": scale_pos_weight,
    "tree_method": "hist",
    "n_jobs": -1,
}


# ==========================================
# FUNÇÃO OBJETIVO DO OPTUNA
# ==========================================


def objective(trial):
    # ======================================
    # HIPERPARÂMETROS
    # ======================================

    trial_params = {
        "n_estimators": trial.suggest_int(
            name="n_estimators",
            low=200,
            high=800,
            step=50,
        ),
        "max_depth": trial.suggest_int(
            name="max_depth",
            low=2,
            high=6,
        ),
        "learning_rate": trial.suggest_float(
            name="learning_rate",
            low=0.01,
            high=0.20,
            log=True,
        ),
        "min_child_weight": trial.suggest_int(
            name="min_child_weight",
            low=1,
            high=12,
        ),
        "subsample": trial.suggest_float(
            name="subsample",
            low=0.60,
            high=1.00,
            step=0.05,
        ),
        "colsample_bytree": trial.suggest_float(
            name="colsample_bytree",
            low=0.60,
            high=1.00,
            step=0.05,
        ),
        "reg_alpha": trial.suggest_float(
            name="reg_alpha",
            low=0.0001,
            high=5.0,
            log=True,
        ),
        "reg_lambda": trial.suggest_float(
            name="reg_lambda",
            low=0.01,
            high=20.0,
            log=True,
        ),
    }

    # ======================================
    # CHILD RUN
    # ======================================

    with mlflow.start_run(
        run_name=(f"trial_{trial.number:03d}"),
        nested=True,
    ):
        # ==================================
        # MODELO
        # ==================================

        trial_model = XGBClassifier(
            **FIXED_PARAMS,
            **trial_params,
        )

        # ==================================
        # TREINAMENTO
        # ==================================

        trial_model.fit(
            X_train,
            y_train,
        )

        # ==================================
        # PROBABILIDADE DA VALIDAÇÃO
        # ==================================

        valid_probability = trial_model.predict_proba(X_valid)[:, 1]

        # ==================================
        # MÉTRICA OBJETIVO
        # ==================================

        valid_pr_auc = average_precision_score(
            y_true=(y_valid),
            y_score=(valid_probability),
        )

        # ==================================
        # PARÂMETROS DA TRIAL
        # ==================================

        mlflow.log_params(
            {
                **trial_params,
                "objective": FIXED_PARAMS["objective"],
                "eval_metric": FIXED_PARAMS["eval_metric"],
                "random_state": RANDOM_STATE,
                "tree_method": FIXED_PARAMS["tree_method"],
                "scale_pos_weight": scale_pos_weight,
            }
        )

        # ==================================
        # MÉTRICA DA TRIAL
        # ==================================

        mlflow.log_metric(
            key=("valid_pr_auc"),
            value=(float(valid_pr_auc)),
        )

        # ==================================
        # TAGS DA TRIAL
        # ==================================

        mlflow.set_tags(
            {
                "project": "reactivation-customers-model",
                "run_role": "optuna_trial",
                "model_family": "xgboost",
                "optimization_metric": "valid_pr_auc",
            }
        )

    return valid_pr_auc

In [0]:
# ==========================================
# RUN PRINCIPAL
# ==========================================

with mlflow.start_run(run_name=RUN_NAME) as run:
    # ======================================
    # TAGS PRINCIPAIS
    # ======================================

    mlflow.set_tags(
        {
            "project": "reactivation-customers-model",
            "model_type": "candidate",
            "model_family": "xgboost",
            "framework": "xgboost",
            "optimization_library": "optuna",
            "class_imbalance_strategy": "scale_pos_weight",
            "dataset": ("workspace.gold_layer.customer_reactivation_modeling"),
        }
    )

    # ======================================
    # ESTUDO DO OPTUNA
    # ======================================

    sampler = optuna.samplers.TPESampler(seed=(RANDOM_STATE))

    study = optuna.create_study(
        direction=("maximize"),
        sampler=(sampler),
        study_name=("xgboost_reactivation_optimization"),
    )

    # ======================================
    # OTIMIZAÇÃO
    # ======================================

    study.optimize(
        func=(objective),
        n_trials=(N_TRIALS),
        n_jobs=(1),
    )

    # ======================================
    # MELHORES HIPERPARÂMETROS
    # ======================================

    best_trial_params = study.best_params

    best_model_params = {
        **FIXED_PARAMS,
        **best_trial_params,
    }

    # ======================================
    # TREINAMENTO DO MELHOR MODELO
    # ======================================

    best_xgb_model = XGBClassifier(**best_model_params)

    best_xgb_model.fit(
        X_train,
        y_train,
    )

    # ======================================
    # AVALIAÇÃO
    # ======================================

    train_metrics, train_confusion = evaluate_classifier(
        model=(best_xgb_model),
        X=(X_train),
        y=(y_train),
        dataset_name=("train"),
    )

    valid_metrics, valid_confusion = evaluate_classifier(
        model=(best_xgb_model),
        X=(X_valid),
        y=(y_valid),
        dataset_name=("valid"),
    )

    test_metrics, test_confusion = evaluate_classifier(
        model=(best_xgb_model),
        X=(X_test),
        y=(y_test),
        dataset_name=("test"),
    )

    all_metrics = {
        **train_metrics,
        **valid_metrics,
        **test_metrics,
    }

    # ======================================
    # PARÂMETROS DO MODELO FINAL
    # ======================================

    mlflow.log_params(
        {
            "model_class": "XGBClassifier",
            "optimization_library": "optuna",
            "optimization_metric": "valid_pr_auc",
            "number_trials": N_TRIALS,
            "objective": FIXED_PARAMS["objective"],
            "eval_metric": FIXED_PARAMS["eval_metric"],
            "random_state": RANDOM_STATE,
            "tree_method": FIXED_PARAMS["tree_method"],
            "scale_pos_weight": scale_pos_weight,
            "negative_class_count": negative_class_count,
            "positive_class_count": positive_class_count,
            "target_column": TARGET_COLUMN,
            "number_features": len(FEATURE_COLUMNS),
            "feature_columns": ", ".join(FEATURE_COLUMNS),
            "top_k_values": "10%, 20%, 30%",
            "top_k_tie_breaker": "random",
            "top_k_random_state": RANDOM_STATE,
            "train_start_date": str(df_train_pandas["reference_date"].min()),
            "train_end_date": str(df_train_pandas["reference_date"].max()),
            "valid_start_date": str(df_valid_pandas["reference_date"].min()),
            "valid_end_date": str(df_valid_pandas["reference_date"].max()),
            "test_start_date": str(df_test_pandas["reference_date"].min()),
            "test_end_date": str(df_test_pandas["reference_date"].max()),
            **best_trial_params,
        }
    )

    # ======================================
    # VOLUMES
    # ======================================

    mlflow.log_metrics(
        {
            "train_rows": float(len(X_train)),
            "valid_rows": float(len(X_valid)),
            "test_rows": float(len(X_test)),
        }
    )

    # ======================================
    # RESULTADOS DO OPTUNA
    # ======================================

    mlflow.log_metrics(
        {
            "optuna_best_valid_pr_auc": float(study.best_value),
            "optuna_best_trial_number": float(study.best_trial.number),
        }
    )

    # ======================================
    # MÉTRICAS COMPLETAS
    # ======================================

    mlflow.log_metrics(all_metrics)

    # ======================================
    # MATRIZES DE CONFUSÃO
    # ======================================

    mlflow.log_dict(
        {
            "train": train_confusion,
            "valid": valid_confusion,
            "test": test_confusion,
        },
        ("evaluation/confusion_matrices.json"),
    )

    # ======================================
    # MELHORES PARÂMETROS
    # ======================================

    mlflow.log_dict(
        best_model_params,
        ("optuna/best_parameters.json"),
    )

    # ======================================
    # RESULTADO DO ESTUDO
    # ======================================

    mlflow.log_dict(
        {
            "study_name": study.study_name,
            "direction": str(study.direction),
            "number_trials": len(study.trials),
            "best_trial_number": study.best_trial.number,
            "best_valid_pr_auc": float(study.best_value),
        },
        ("optuna/study_summary.json"),
    )

    # ======================================
    # HISTÓRICO DAS TRIALS
    # ======================================

    trials_dataframe = study.trials_dataframe(
        attrs=(
            "number",
            "value",
            "params",
            "state",
        )
    )

    trials_dataframe["state"] = trials_dataframe["state"].astype(str)

    mlflow.log_table(
        data=(trials_dataframe),
        artifact_file=("optuna/trials.json"),
    )

    # ======================================
    # METADADOS DAS COLUNAS
    # ======================================

    mlflow.log_dict(
        {
            "identifier_columns": IDENTIFIER_COLUMNS,
            "feature_columns": FEATURE_COLUMNS,
            "target_column": TARGET_COLUMN,
        },
        ("metadata/model_columns.json"),
    )

    # ======================================
    # ASSINATURA
    # ======================================

    model_signature = infer_signature(
        model_input=(X_train),
        model_output=(best_xgb_model.predict(X_train)),
    )

    # ======================================
    # MODELO
    # ======================================

    mlflow.xgboost.log_model(
        xgb_model=(best_xgb_model),
        name=("model"),
        signature=(model_signature),
        input_example=(X_train.head(5)),
        model_format=("json"),
    )

    # ======================================
    # RESULTADO
    # ======================================

    print(
        f"""
        Otimização concluída com sucesso.

        Melhor trial:

        {study.best_trial.number}

        Melhor PR-AUC de validação:

        {study.best_value:.6f}

        Melhores parâmetros:

        {study.best_params}

        Run ID principal:

        {run.info.run_id}
        """
    )

In [0]:
y_train_pred = best_xgb_model.predict(X_train)
y_valid_pred = best_xgb_model.predict(X_valid)
y_test_pred = best_xgb_model.predict(X_test)

In [0]:
display_confusion_matrix(y_true=y_train, y_pred=y_train_pred, dataset_name="Train")

display_confusion_matrix(y_true=y_valid, y_pred=y_valid_pred, dataset_name="Validation")

display_confusion_matrix(y_true=y_test, y_pred=y_test_pred, dataset_name="Test")